# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [1]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display

from _bootstrap import ROOT
from utils import BaselineRouteGenerator, JeepneySystem, JeepneyRouteEnv

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


,route_id,anchors,area_m2,length_m,attempt
0,B401,"[1681, 123, 2663, 253]",50877467.0,68122.0,1
1,B402,"[62, 304, 452, 826]",20744262.0,28657.0,1
2,B403,"[2189, 2467, 34, 975]",55598071.0,79287.0,1
3,B404,"[366, 1778, 908, 1110]",62670587.0,71579.0,1
4,B405,"[2440, 1683, 1795, 32]",9824230.0,36110.0,1
5,B406,"[1350, 145, 631, 24]",7065176.0,23928.0,1
6,B407,"[2161, 1110, 171, 2560]",16543363.0,47544.0,1
7,B408,"[1093, 62, 3237, 1194]",33425046.0,41603.0,1
8,B409,"[8, 1921, 1939, 1184]",2176606.0,17352.0,1
9,B410,"[2888, 786, 772, 523]",7180967.0,62373.0,1


## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)

def print_observation(obs: dict[str, np.ndarray]) -> None:
    print('shape:', obs['shape'])
    print('history:', obs['history'])
    print('topology:', obs['topology'])
    print('demand:', obs['demand'])
    print('global:', obs['global'])
    print('candidates:')
    for candidate_index, row in enumerate(obs['candidates']):
        print(f'  {candidate_index}: {row}')
    print('mask:', obs['action_mask'])

obs, info = env.reset()
print('reset state vector length:', info['state_vector'].shape[0])
print_observation(obs)

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('turn angle rad:', info['turn_angle_rad'])
    print('sinuosity index:', info['sinuosity_index'])
    print('distance to origin m:', info['distance_to_origin_m'])
    print('bearing to origin rad:', info['bearing_to_origin_rad'])
    print('route area m2:', info['route_area_m2'])
    print('state vector:', info['state_vector'])
    print_observation(obs)
    if terminated or truncated:
        break


reset state vector length: 91
shape: [0. 0. 1. 0. 0. 0. 0.]
history: [0. 0. 0. 0. 0. 0. 0. 0.]
topology: [0.5 0.5 0.  0.5 0.5]
demand: [0.298 0.289 0.348 0.051]
global: [0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.5   0.5   0.    0.5   0.5   0.298 0.289 0.348 0.051]
candidates:
  0: [-0.379 -0.925  0.011  0.5    0.     0.348  0.051  0.     1.     0.   ]
  1: [-0.202  0.979  0.033  0.5    0.     0.307  0.01   0.     0.     0.   ]
  2: [ 0.323  0.947  0.023  0.5    0.     0.21  -0.087  0.     0.     0.   ]
  3: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  4: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  5: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
mask: [1. 1. 1. 0. 0. 0. 1.]
step 1: action=1, reward=-0.000, terminated=False, truncated=False
turn angle rad: 0.0
sinuosity index: 1.0
distance to origin m: 224.39023217927448
bearing to origin rad: 3.141592653589793
route area m2: 0.0
state vector: [ 0.003  0.    -1.     0.003  0.003  0.     0.     0.     0.     0.
  0.     1.  

In [3]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

NOTE_VERBOSITY = "succinct"

def build_route_encoding(row: pd.Series, area: float, length: float, area_to_length: float) -> str:
    return (
        f"route_id: {row.route_id}\n"
        f"anchors: {list(row.anchors)}\n"
        f"area_m2: {area:,.0f}\n"
        f"length_m: {length:,.0f}\n"
        f"attempt: {int(row.attempt)}\n"
        f"area_to_length: {area_to_length:.2f}"
    )

def interpret_route_geometry(*, area: float, length: float, area_median: float, length_median: float, verbosity: str = NOTE_VERBOSITY) -> str:
    area_to_length = area / max(length, 1.0)
    if area >= area_median and length <= length_median:
        interpretation = "Strong candidate: wide coverage without much extra travel, so this route is likely efficient."
    elif area >= area_median and length > length_median:
        interpretation = "Broad reach, but the route is paying for it in distance; useful coverage, less efficient movement."
    elif area < area_median and length >= length_median:
        interpretation = "This looks overextended for its footprint, which can signal wasted traversal or a corridor bias."
    else:
        interpretation = "Tight local loop: probably practical for neighborhood service, but not a wide-area connector."

    if verbosity == "verbose":
        interpretation += f" Scale cue: area-to-length={area_to_length:.2f}."
    return interpretation

def build_route_notes(summary_df: pd.DataFrame, *, verbosity: str = NOTE_VERBOSITY) -> dict[str, dict[str, str]]:
    area_median = float(summary_df['area_m2'].median())
    length_median = float(summary_df['length_m'].median())
    notes: dict[str, dict[str, str]] = {}
    for row in summary_df.itertuples(index=False):
        area = float(row.area_m2)
        length = float(row.length_m)
        area_to_length = area / max(length, 1.0)
        notes[row.route_id] = {
            "encoding": build_route_encoding(row, area, length, area_to_length),
            "interpretation": interpret_route_geometry(
                area=area,
                length=length,
                area_median=area_median,
                length_median=length_median,
                verbosity=verbosity,
            ),
        }
    return notes

route_notes = build_route_notes(summary, verbosity=NOTE_VERBOSITY)

html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\results\B4_baseline_routes\B4_route_explorer.html
